# Stage 2.5 starvation workload-visibility repair

This handoff compares frozen-BC/P-final control with the default-off starvation workload-visibility repair. Persistent queues, queue ownership repair, schedule-informed hiring, upkeep, wheat, underfoot, and deadline flags are held equal. The known queue-failure seed is captured from the full ordered panel so episode IDs remain stable. No result is valid until the source/checkpoint/engine preflight passes.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO = Path(os.environ.get('KAGGRICULTURE_REPO', '/kaggle/working/Kaggriculture'))
PPO_CHECKPOINT = Path(os.environ['PPO_CHECKPOINT'])
BC_E_CHECKPOINT = Path(os.environ['BC_E_CHECKPOINT'])
CODE_SHA = os.environ['STAGE25_CODE_SHA']
KNOWN_SEED = 1470672056
SEEDS = [144368101, 309507, 615013, 918079, 1221109, 1524137, 1827169, 2130193, 2433221, 2736251, 3039283, 3342311, 3645341, 3948373, 4251401, 2112243121, 1470672056, 995106988, 1303793286, 521973470, 107449192, 768565387, 1370134739, 2090797777, 425789796, 1027359148, 1688475343, 261654734, 863224086, 1524340281, 97519672, 699089024]
for path in (REPO, PPO_CHECKPOINT, BC_E_CHECKPOINT):
    if not path.exists(): raise FileNotFoundError(path)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual == CODE_SHA, (actual, CODE_SHA)
assert len(SEEDS) == 32 and SEEDS[16] == KNOWN_SEED
print({'source_commit': actual, 'seeds': len(SEEDS), 'processes': 4, 'schedule_informed_hiring': False})

In [ ]:
import datetime
ROOT = Path('/kaggle/working') / ('stage25_starvation_visibility_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S'))
ROOT.mkdir()
COMMON = ['--checkpoint', str(PPO_CHECKPOINT), '--e-checkpoint', str(BC_E_CHECKPOINT), '--backend', 'official', '--e-history-version', 'E_LEGACY', '--master-seed', '25', '--processes', '4', '--variants', 'combined', 'combined_wheat3', '--underfoot-first', '--deadline-safe-planting', '--deadline-safe-hiring', '--seeds', *map(str, SEEDS)]
def run_arm(name, repair, filters=None, capture=True):
    output = ROOT / name
    command = [sys.executable, '-m', 'tools.run_stage25_upkeep_sharded', *COMMON, '--output-dir', str(output)]
    if capture: command += ['--capture-dir', str(ROOT / (name + '-capture'))]
    if filters: command += ['--game-filters', *filters]
    if repair: command += ['--starvation-workload-visibility-repair']
    (ROOT / (name + '.command.json')).write_text(json.dumps({'command': command, 'source_commit': CODE_SHA, 'persistent_worker_queues': False, 'queue_ownership_repair': False, 'schedule_informed_hiring': False}, indent=2))
    subprocess.run(command, cwd=REPO, check=True)
run_arm('known-control', False, [f'{KNOWN_SEED}:0', f'{KNOWN_SEED}:1'])
run_arm('known-repair', True, [f'{KNOWN_SEED}:0', f'{KNOWN_SEED}:1'])
run_arm('panel-control', False, capture=False)
run_arm('panel-repair', True, capture=False)
print(ROOT)

## Reporting

Inspect the targeted captures before the panel summary. Compare feasible maintenance coverage, critical-feed completion time, late-hire bursts and marginal costs, crop-to-weed transitions, completed useful work, both banks, and margin. Task visibility alone is not evidence that the frozen day-16 crop loss is fixed. Preserve `manifest.json`, `games.jsonl`, capture audits, command JSON, checkpoint hashes, and engine provenance.